<small>Reconocimiento de Patrones · Licenciatura en Ciencias de la Computación · Universidad de Sonora<br>
Julio Waissman · <a href="https://ml-unison.github.io">ml-unison.github.io</a> · Licencia CC BY-SA 4.0</small>

---

# Laboratorio 2 · k-medias y PCA desde cero

**Unidad 2**

En clase usamos `KMeans` y `PCA` de scikit-learn como cajas negras. En este laboratorio vas a escribir los dos algoritmos tú mismo, solo con numpy, a comprobar que dan lo mismo que scikit-learn y a usarlos en dos problemas concretos: cuántas dimensiones hacen falta para describir imágenes de dígitos escritos a mano y cómo reducir una fotografía a 16 colores.

**Cómo se trabaja.** Completa las celdas marcadas con `...`. Después de cada pregunta hay una celda `grader.check("p1")` que verifica tu respuesta: si algo falla, te dice qué. Las preguntas marcadas como *reflexión* se responden por escrito en la celda indicada.

**Reglas.** Dentro de tus funciones usa solo numpy: nada de `sklearn` (scikit-learn se usa únicamente para comparar). Evita los ciclos sobre los puntos: las operaciones deben ser vectorizadas. Un ciclo sobre los $k$ grupos o sobre las iteraciones del algoritmo sí está permitido.

**Cómo se entrega.** Ejecuta la libreta de principio a fin (*Entorno de ejecución → Reiniciar y ejecutar todo*), revisa que no haya errores y descarga el archivo `.ipynb`.

Las verificaciones de esta libreta son solo una parte de la revisión: al calificar se usan pruebas adicionales.

In [ ]:
import os, sys

RAMA = "main"  # rama del repositorio del curso de la que se toman paquete, datos y pruebas
os.environ["RDP_RAMA"] = RAMA
if "google.colab" in sys.modules:
    %pip install -q otter-grader "rdp @ git+https://github.com/ml-unison/ml-unison.github.io@{RAMA}#subdirectory=paquete"

In [ ]:
# Verificador de respuestas (otter-grader). Usa las pruebas públicas del laboratorio:
# la copia local si existe; si no (en Colab), las descarga del repositorio del curso.
import pathlib, urllib.request
import otter

PRUEBAS = pathlib.Path("tests") / "u02-lab02-kmedias-pca"
if not PRUEBAS.exists():
    PRUEBAS.mkdir(parents=True)
    for _p in ['p1.py', 'p2.py', 'p3.py', 'p4.py', 'p5.py', 'p6.py', 'p7.py']:
        urllib.request.urlretrieve(
            f"https://raw.githubusercontent.com/ml-unison/ml-unison.github.io/main/unidades/02-no-supervisado/laboratorios/tests/u02-lab02-kmedias-pca/{_p}",
            PRUEBAS / _p)
grader = otter.Notebook(tests_dir=str(PRUEBAS))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs, load_digits, load_sample_image
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

import rdp

rdp.estilo.usar()

## Los datos

Usaremos tres conjuntos, todos incluidos en scikit-learn (no hay que descargar nada):

- **Grupos sintéticos** generados con `make_blobs`: 600 puntos en el plano alrededor de 5 centros. Como sabemos cómo se generaron, sirven para comprobar que el algoritmo funciona.
- **Dígitos escritos a mano** (`load_digits`): 1,797 imágenes de 8 × 8 píxeles en escala de grises, tomadas del repositorio UCI (*Optical Recognition of Handwritten Digits*). Cada imagen es un vector de 64 números.
- **Una fotografía** (`load_sample_image("china.jpg")`): un templo en China, de 427 × 640 píxeles en color, distribuida con scikit-learn bajo licencia CC BY 4.0.

**Problemas.** (1) Implementar k-medias y PCA y verificarlos. (2) Averiguar cuántas componentes principales conservan el 90 % de la varianza de los dígitos. (3) *Cuantizar* la fotografía: representarla con solo 16 colores elegidos por k-medias.

In [ ]:
X, _ = make_blobs(n_samples=600, centers=5, cluster_std=1.2, random_state=1)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(X[:, 0], X[:, 1], s=12, color=rdp.estilo.GRIS, alpha=0.7)
ax.set_title("600 puntos sin etiquetas: ¿cuántos grupos ves?")
ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$");

## 1. k-medias, pieza por pieza

Recordemos el algoritmo de Lloyd. Tenemos $n$ puntos $x_1, \dots, x_n \in \mathbb{R}^d$ (las filas de una matriz `X` de $n \times d$) y $k$ centroides $c_1, \dots, c_k$ (las filas de una matriz `C` de $k \times d$). Se repiten dos pasos:

1. **Asignar**: cada punto va al grupo de su centroide más cercano, $\ell_i = \arg\min_j \lVert x_i - c_j \rVert^2$.
2. **Actualizar**: cada centroide se mueve a la media de los puntos de su grupo.

Lo que se minimiza es la **inercia** (suma de distancias al cuadrado de cada punto a su centroide):

$$J = \sum_{i=1}^{n} \lVert x_i - c_{\ell_i} \rVert^2 .$$

**Pregunta 1.** Escribe `asignar(X, C)`, que devuelve un arreglo de enteros de longitud $n$ con el índice del centroide más cercano a cada fila de `X`. Si un punto está a la misma distancia de dos centroides, se queda con el de **menor índice**.

Hazlo sin ciclos: con *broadcasting*, `X[:, None, :] - C[None, :, :]` tiene forma $n \times k \times d$.

In [ ]:
def asignar(X, C):
    """Índice del centroide más cercano (distancia euclidiana) para cada fila de X."""
    ...


asignar(np.array([[0.0, 0.0], [5.0, 5.0], [4.0, 6.0]]), np.array([[0.0, 1.0], [5.0, 5.0]]))

In [ ]:
grader.check("p1")

**Pregunta 2.** Escribe `actualizar(X, etiquetas, C)`, que devuelve una **nueva** matriz de centroides de $k \times d$ (con $k$ = número de filas de `C`):

- la fila $j$ es la **media** de los puntos con `etiquetas == j`;
- si ningún punto quedó en el grupo $j$ (grupo vacío), la fila $j$ conserva el centroide anterior `C[j]`.

No modifiques `C`: haz una copia. Puedes usar un ciclo sobre los $k$ grupos.

In [ ]:
def actualizar(X, etiquetas, C):
    """Nuevos centroides: media de cada grupo; un grupo vacío conserva su centroide anterior."""
    ...


actualizar(np.array([[0.0, 0.0], [2.0, 0.0], [10.0, 10.0]]), np.array([0, 0, 1]), np.zeros((2, 2)))

In [ ]:
grader.check("p2")

**Pregunta 3.** Escribe `inercia(X, C, etiquetas)`, que devuelve (como `float`) la suma de las distancias al cuadrado de cada punto **al centroide que indican sus etiquetas**. Ojo: usa las etiquetas que recibes, aunque no sean las del centroide más cercano. Pista: `C[etiquetas]` es una matriz de $n \times d$.

In [ ]:
def inercia(X, C, etiquetas):
    """Suma de distancias al cuadrado de cada punto al centroide de su grupo."""
    ...


inercia(np.array([[0.0, 0.0], [2.0, 0.0]]), np.array([[1.0, 0.0]]), np.array([0, 0]))

In [ ]:
grader.check("p3")

**Pregunta 4.** Junta las piezas en `kmedias(X, k, semilla, max_iter, tol)`. La inicialización ya está escrita: elige $k$ filas distintas de `X` al azar con un generador de semilla fija. Tú escribe el ciclo; en cada iteración:

1. asigna los puntos a los centroides actuales `C`;
2. agrega a `historial` la inercia de esa asignación con esos centroides;
3. calcula los centroides nuevos con `actualizar`;
4. si ningún centroide se movió más de `tol` en ninguna coordenada (`np.abs(nuevos - C).max() < tol`), reemplaza `C` y termina; si no, reemplaza `C` y sigue.

Haz a lo más `max_iter` iteraciones. La función devuelve los centroides finales, las etiquetas que les corresponden y el historial de inercias.

In [ ]:
def kmedias(X, k, semilla=0, max_iter=100, tol=1e-6):
    '''k-medias (algoritmo de Lloyd). Devuelve (C, etiquetas, historial).'''
    X = np.asarray(X, dtype=float)
    rng = np.random.default_rng(semilla)
    C = X[rng.choice(len(X), size=k, replace=False)]
    historial = []
    ...
    return C, asignar(X, C), historial


C, etiquetas, historial = kmedias(X, 5, semilla=0)
print(f"{len(historial)} iteraciones · inercia final: {historial[-1]:.1f}")

In [ ]:
grader.check("p4")

Veamos el historial de inercias para diez semillas distintas. Todas las curvas bajan, pero ¿llegan todas al mismo lugar?

In [ ]:
inercia_sklearn = KMeans(n_clusters=5, n_init=10, random_state=0).fit(X).inertia_
fig, ax = plt.subplots(figsize=(8, 4.5))
for s in range(10):
    _, _, h = kmedias(X, 5, semilla=s)
    ax.plot(range(1, len(h) + 1), h, marker="o", markersize=3, color=rdp.estilo.AZUL, alpha=0.6)
ax.axhline(inercia_sklearn, color=rdp.estilo.ROJO, ls="--")
ax.set_ylim(bottom=inercia_sklearn * 0.9)
ax.text(ax.get_xlim()[1], inercia_sklearn * 0.98, "scikit-learn (10 inicializaciones) ", color=rdp.estilo.ROJO,
        ha="right", va="top")
ax.set_yscale("log")
ax.set_title("Inercia por iteración, una curva por semilla")
ax.set_xlabel("iteración")
ax.set_ylabel("inercia (escala log)");

## 2. PCA con la descomposición en valores singulares

Si centramos los datos ($\tilde X = X - \bar x$) y calculamos la descomposición en valores singulares $\tilde X = U \Sigma V^\top$, las **direcciones principales** son las filas de $V^\top$, ordenadas de mayor a menor valor singular $\sigma_1 \ge \sigma_2 \ge \dots$ La varianza que captura la componente $j$ es proporcional a $\sigma_j^2$, así que la **proporción de varianza explicada** es

$$\frac{\sigma_j^2}{\sum_i \sigma_i^2},$$

y la proyección de los datos sobre las primeras $m$ componentes es $\tilde X V_m$, donde $V_m$ son las primeras $m$ columnas de $V$.

**Pregunta 5.** Escribe `pca(X, m)`, que devuelve una tupla `(componentes, proporcion, Z)`:

- `componentes`: matriz de $m \times d$ con las $m$ direcciones principales (una por fila);
- `proporcion`: arreglo de longitud $m$ con la proporción de varianza explicada por cada una;
- `Z`: matriz de $n \times m$ con los datos centrados proyectados sobre esas direcciones.

Usa `np.linalg.svd(..., full_matrices=False)`. El signo de cada componente es arbitrario (tanto $v$ como $-v$ son válidas): las pruebas lo toman en cuenta.

In [ ]:
def pca(X, m):
    """PCA vía SVD. Devuelve (componentes m×d, proporción de varianza explicada, X proyectada n×m)."""
    ...


componentes, proporcion, Z = pca(X, 2)
proporcion

In [ ]:
grader.check("p5")

## 3. ¿Cuántas dimensiones tienen los dígitos?

Cada dígito es un vector de 64 intensidades, pero intuitivamente no hacen falta 64 números para describir un «3»: los píxeles vecinos están muy correlacionados y los de las orillas casi siempre valen cero.

In [ ]:
digitos = load_digits().data
fig, axes = plt.subplots(2, 8, figsize=(10, 3))
for imagen, ax in zip(digitos[:16], axes.ravel()):
    ax.imshow(imagen.reshape(8, 8), cmap="gray_r")
    ax.axis("off")
fig.suptitle("Los primeros 16 dígitos: imágenes de 8 × 8 píxeles");

**Pregunta 6.** Usa **tu** función `pca` sobre `digitos` para calcular `n_90`: el **mínimo** número de componentes cuya varianza explicada acumulada es al menos 90 %. Debe ser un entero. Pista: `np.cumsum`.

In [ ]:
_, proporcion_digitos, _ = pca(digitos, 64)
n_90 = ...
n_90

In [ ]:
grader.check("p6")

In [ ]:
acumulada = np.cumsum(proporcion_digitos)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(range(1, 65), acumulada, color=rdp.estilo.AZUL, marker="o", markersize=3)
ax.axhline(0.90, color=rdp.estilo.GRIS, ls="--")
ax.text(64, 0.885, "90 %", color=rdp.estilo.TENUE, ha="right", va="top")
ax.set_title("Varianza explicada acumulada en los dígitos")
ax.set_xlabel("número de componentes")
ax.set_ylabel("proporción acumulada");

## 4. Aplicación: una fotografía con 16 colores

Una fotografía en color guarda tres números por píxel (rojo, verde y azul), así que puede tener millones de colores distintos. Si tratamos cada píxel como un punto de $\mathbb{R}^3$ y lo agrupamos con k-medias, los $k$ centroides son una **paleta** de $k$ colores; reemplazar cada píxel por su centroide da una imagen que solo usa esos colores. Así funcionaban los formatos de imagen con paleta, como GIF.

In [ ]:
imagen = load_sample_image("china.jpg") / 255  # valores entre 0 y 1
pixeles = imagen.reshape(-1, 3)
print(f"{imagen.shape[0]} × {imagen.shape[1]} píxeles · {len(np.unique(pixeles, axis=0)):,} colores distintos")

**Pregunta 7.** Agrupar los 273,280 píxeles es innecesario: basta una muestra.

1. Toma una muestra de 5,000 píxeles **sin reemplazo** con `np.random.default_rng(0)` y guárdala en `muestra`.
2. Corre tu `kmedias` sobre `muestra` con `k=16` y `semilla=0`; guarda los centroides en `colores` ($16 \times 3$).
3. Asigna **todos** los píxeles a su color más cercano y construye `cuantizada`: un arreglo con la misma forma que `imagen` en el que cada píxel se reemplaza por su color.
4. Calcula `ecm`, el error cuadrático medio entre `cuantizada` e `imagen` (promedio sobre todos los píxeles y canales).

In [ ]:
muestra = ...
colores = ...
cuantizada = ...
ecm = ...
print(f"error cuadrático medio: {ecm:.5f}")

In [ ]:
grader.check("p7")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].imshow(imagen)
axes[0].set_title(f"Original: {len(np.unique(pixeles, axis=0)):,} colores")
axes[1].imshow(cuantizada)
axes[1].set_title("Cuantizada con k-medias: 16 colores")
for ax in axes:
    ax.axis("off")

Y la paleta que encontró k-medias:

In [ ]:
fig, ax = plt.subplots(figsize=(10, 1.2))
ax.imshow(colores[None, :, :], aspect="auto")
ax.set_axis_off()
ax.set_title("Los 16 centroides, vistos como colores");

## 5. Reflexión

Estas preguntas no se verifican automáticamente: respóndelas en la celda de texto que sigue a cada una.

<!-- BEGIN QUESTION -->

**Reflexión 1.** En la gráfica del historial, ¿todas las semillas llegaron a la misma inercia final? Explica por qué k-medias puede quedarse atorado en una solución peor (piensa en qué garantiza cada paso del algoritmo y qué no) y por qué en la práctica se corre con varias inicializaciones (`n_init` en scikit-learn), quedándose con la de menor inercia.

_Escribe aquí tu respuesta._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

**Reflexión 2.** En la cuantización, ¿qué representa cada centroide y qué representa la inercia? En clase dijimos que antes de agrupar casi siempre conviene estandarizar los atributos. ¿Tendría sentido aquí estandarizar los canales rojo, verde y azul? Argumenta qué cambiaría en la distancia entre colores y en el resultado.

_Escribe aquí tu respuesta._

<!-- END QUESTION -->



---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
grader.check_all()